In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('/content/train.csv', engine='python', on_bad_lines='skip')


In [ ]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [ ]:
df.isnull().sum()

,0
id,0
qid1,0
qid2,0
question1,1
question2,2
is_duplicate,0


In [ ]:
df=df.dropna()

In [ ]:
df.isnull().sum()

,0
id,0
qid1,0
qid2,0
question1,0
question2,0
is_duplicate,0


In [ ]:
df=df.drop(['id','qid1','qid2'],axis=1)

In [ ]:
df.head()

,question1,question2,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [ ]:
df['question2'][3]

'Find the remainder when [math]23^{24}[/math] is divided by 24,23?'

In [ ]:
new_df = df.sample(80000,random_state=2)

In [ ]:
import re
from bs4 import BeautifulSoup

In [ ]:
def preprocess(q):

    q = str(q).lower().strip()

    # Replace certain special characters with their string equivalents
    q = q.replace('%', ' percent')
    q = q.replace('$', ' dollar ')
    q = q.replace('₹', ' rupee ')
    q = q.replace('€', ' euro ')
    q = q.replace('@', ' at ')

    # The pattern '[math]' appears around 900 times in the whole dataset.
    q = q.replace('[math]', '')

    # Replacing some numbers with string equivalents (not perfect, can be done better to account for more cases)
    q = q.replace(',000,000,000 ', 'b ')
    q = q.replace(',000,000 ', 'm ')
    q = q.replace(',000 ', 'k ')
    # q = re.sub(r'[^\w\s+\-*/=<>^%]', ' ', q)

    # Decontracting words
    # https://en.wikipedia.org/wiki/Wikipedia%3aList_of_English_contractions
    # https://stackoverflow.com/a/19794953
    contractions = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "can not",
    "can't've": "can not have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "i would",
    "i'd've": "i would have",
    "i'll": "i will",
    "i'll've": "i will have",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so as",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there would",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you would",
    "you'd've": "you would have",
    "you'll": "you will",
    "you'll've": "you will have",
    "you're": "you are",
    "you've": "you have"
    }

    q_decontracted = []

    for word in q.split():
        if word in contractions:
            word = contractions[word]

        q_decontracted.append(word)

    q = ' '.join(q_decontracted)
    q = q.replace("'ve", " have")
    q = q.replace("n't", " not")
    q = q.replace("'re", " are")
    q = q.replace("'ll", " will")


    # Removing HTML tags
    q = BeautifulSoup(q)
    q = q.get_text()
    q = re.sub(r'[^\w\s]', ' ', q)
    q = re.sub(r'\s+', ' ', q).strip()

    return q

In [ ]:
new_df['question1']=new_df['question1'].apply(preprocess)
new_df['question2']=new_df['question2'].apply(preprocess)

In [ ]:
new_df['question2'][3]

'find the remainder when 23 24 math is divided by 24 23'

In [ ]:
df.head()

,question1,question2,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X=new_df.drop(['is_duplicate'],axis=1)
y=new_df['is_duplicate']

In [ ]:
print(X)

                                                question1  \
331535                          how can i learn norwegian   
45407                   how are currency rates determined   
286200                               what is substitution   
157195       how can i make iphone 4s faster with ios 9 2   
154346  how can i help my girlfriend cope with her par...   
...                                                   ...   
37804             which are the best jeans bands in india   
51900                       what was world like in 1880 s   
267014  why do people want to become rich in terms of ...   
133948                   how do you block people on quora   
22346   what is the level of difficulty to get a job i...   

                                                question2  
331535        what is the quickest way to learn norwegian  
45407         where and how are exchange rates determined  
286200              what is a substitute for caciocavallo  
157195  i have an iphone 4s

In [ ]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,stratify=y)

In [ ]:
vectorizer=TfidfVectorizer()

In [ ]:
import scipy.sparse as sp

# 1. Extract the columns as 1D arrays
train_q1 = X_train['question1'].values.astype('U')
train_q2 = X_train['question2'].values.astype('U')

test_q1 = X_test['question1'].values.astype('U')
test_q2 = X_test['question2'].values.astype('U')

# 2. Combine all training questions into a single list to build a shared vocabulary
train_questions_combined = list(train_q1) + list(train_q2)

# Fit the vectorizer on the combined vocabulary (do NOT fit on test data)
vectorizer.fit(train_questions_combined)

# 3. Transform Q1 and Q2 separately using the shared vectorizer
X_train_q1_vec = vectorizer.transform(train_q1)
X_train_q2_vec = vectorizer.transform(train_q2)

X_test_q1_vec = vectorizer.transform(test_q1)
X_test_q2_vec = vectorizer.transform(test_q2)

# 4. Horizontally stack the matrices side-by-side
# (This puts Q1's features and Q2's features next to each other for the model)
X_train_vec = sp.hstack((X_train_q1_vec, X_train_q2_vec))
X_test_vec = sp.hstack((X_test_q1_vec, X_test_q2_vec))

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Define models
models = {
    # "Naive Bayes": MultinomialNB(),

    # "SVM": LinearSVC(),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    )
}


# Store results
results = []


# Train and evaluate each model
for name, model in models.items():

    print(f"Training {name}...")

    # Train
    model.fit(X_train_vec, y_train)

    # Prediction
    y_pred = model.predict(X_test_vec)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    f1 = f1_score(y_test, y_pred, average='binary')

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })


# Create comparison DataFrame
import pandas as pd

results_df = pd.DataFrame(results)

print(results_df)

Training Random Forest...
Training XGBoost...
           Model  Accuracy  Precision    Recall  F1 Score
0  Random Forest   0.77015   0.779151  0.528386  0.629722
1        XGBoost   0.74075   0.722591  0.485537  0.580807


In [ ]:
final_model=RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

In [ ]:
final_model.fit(X_train_vec,y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
import scipy.sparse as sp

def predict_duplicate(q1, q2, vectorizer, rf_model):
    # 1. Preprocess both questions using your custom function
    clean_q1 = preprocess(q1)
    clean_q2 = preprocess(q2)

    # 2. Vectorize using the PRE-FIT vectorizer
    # The vectorizer expects an iterable (like a list), so we wrap the string in [ ]
    q1_vec = vectorizer.transform([clean_q1])
    q2_vec = vectorizer.transform([clean_q2])

    # 3. Stack them horizontally exactly as you did for the training data
    query_vec = sp.hstack((q1_vec, q2_vec))

    # 4. Predict using the trained Random Forest model
    # [0] extracts the integer from the output array (e.g., [1] -> 1)
    prediction = rf_model.predict(query_vec)[0]

    # Optional: Get the confidence probability for better insights
    probability = rf_model.predict_proba(query_vec)[0]

    if prediction == 1:
        return f"Duplicate (Confidence: {probability[1]:.2%})"
    else:
        return f"Not Duplicate (Confidence: {probability[0]:.2%})"

# ==========================================
# Example Usage
# ==========================================
# Note: 'vectorizer' and 'rf_model' must be the exact objects you fit earlier
question_pairs = [
    # Duplicate / Same meaning
    (
        "How can I learn Python?",
        "What is the best way to learn Python?"
    ),
    (
        "How can I lose weight?",
        "What are the best ways to lose weight?"
    ),
    (
        "What is machine learning?",
        "Can someone explain machine learning?"
    ),
    (
        "How do I learn data science?",
        "What is the best way to learn data science?"
    ),
    (
        "How can I improve my English?",
        "What are the best ways to improve my English?"
    ),

    # Not duplicate / Different meaning
    (
        "How can I learn Python?",
        "How can I learn Java?"
    ),
    (
        "What is machine learning?",
        "What is deep learning?"
    ),
    (
        "How can I lose weight?",
        "How can I gain weight?"
    ),
    (
        "What is the capital of India?",
        "What is the population of India?"
    ),
    (
        "How do I learn data science?",
        "How do I get a job in data science?"
    )
]

for q1, q2 in question_pairs:
    print(predict_duplicate(q1, q2, vectorizer, final_model))

# question_1 = "What is the best way to learn machine learning?"
# question_2 = "How do I get started with machine learning?"

# result = predict_duplicate(question_1, question_2, vectorizer, final_model)
# print(result)

Not Duplicate (Confidence: 67.50%)
Duplicate (Confidence: 52.50%)
Not Duplicate (Confidence: 63.50%)
Not Duplicate (Confidence: 70.00%)
Duplicate (Confidence: 86.50%)
Duplicate (Confidence: 67.00%)
Duplicate (Confidence: 56.50%)
Duplicate (Confidence: 59.00%)
Not Duplicate (Confidence: 60.50%)
Not Duplicate (Confidence: 55.50%)
